In [1]:
import pandas as pd
import duckdb

transactions = pd.DataFrame({
    "customer_id": ["C1", "C1", "C2", "C2", "C3", "C1"],
    "product_id": ["P1", "P2", "P1", "P3", "P2", "P1"],
    "amount": [50, 30, 20, 80, 40, 60],
    "txn_date": pd.to_datetime(["2026-01-05", "2026-01-10", "2026-01-07",
                                  "2026-01-15", "2026-01-20", "2026-01-25"])
})

products = pd.DataFrame({
    "product_id": ["P1", "P2", "P4"],
    "category": ["Electronics", "Books", "Toys"]
})

duckdb.sql("CREATE VIEW transactions_v AS SELECT * FROM transactions")
duckdb.sql("CREATE VIEW products_v AS SELECT * FROM products")

In [2]:
duckdb.sql("""
SELECT t.*, p.category
FROM transactions_v t
LEFT JOIN products_v p ON t.product_id = p.product_id
""").show()

┌─────────────┬────────────┬────────┬─────────────────────┬─────────────┐
│ customer_id │ product_id │ amount │      txn_date       │  category   │
│   varchar   │  varchar   │ int64  │      timestamp      │   varchar   │
├─────────────┼────────────┼────────┼─────────────────────┼─────────────┤
│ C1          │ P1         │     50 │ 2026-01-05 00:00:00 │ Electronics │
│ C1          │ P2         │     30 │ 2026-01-10 00:00:00 │ Books       │
│ C2          │ P1         │     20 │ 2026-01-07 00:00:00 │ Electronics │
│ C3          │ P2         │     40 │ 2026-01-20 00:00:00 │ Books       │
│ C1          │ P1         │     60 │ 2026-01-25 00:00:00 │ Electronics │
│ C2          │ P3         │     80 │ 2026-01-15 00:00:00 │ NULL        │
└─────────────┴────────────┴────────┴─────────────────────┴─────────────┘



In [4]:
duckdb.sql("""
SELECT t.customer_id, SUM(t.amount) , AVG(t.amount) , COUNT(*) 
FROM transactions_v t
GROUP BY t.customer_id
""").show()


┌─────────────┬───────────────┬────────────────────┬──────────────┐
│ customer_id │ sum(t.amount) │   avg(t.amount)    │ count_star() │
│   varchar   │    int128     │       double       │    int64     │
├─────────────┼───────────────┼────────────────────┼──────────────┤
│ C3          │            40 │               40.0 │            1 │
│ C2          │           100 │               50.0 │            2 │
│ C1          │           140 │ 46.666666666666664 │            3 │
└─────────────┴───────────────┴────────────────────┴──────────────┘



In [5]:
duckdb.sql("""
SELECT t.*, 
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) as rank_in_customer
FROM transactions_v t
""").show()

┌─────────────┬────────────┬────────┬─────────────────────┬──────────────────┐
│ customer_id │ product_id │ amount │      txn_date       │ rank_in_customer │
│   varchar   │  varchar   │ int64  │      timestamp      │      int64       │
├─────────────┼────────────┼────────┼─────────────────────┼──────────────────┤
│ C2          │ P3         │     80 │ 2026-01-15 00:00:00 │                1 │
│ C2          │ P1         │     20 │ 2026-01-07 00:00:00 │                2 │
│ C3          │ P2         │     40 │ 2026-01-20 00:00:00 │                1 │
│ C1          │ P1         │     60 │ 2026-01-25 00:00:00 │                1 │
│ C1          │ P1         │     50 │ 2026-01-05 00:00:00 │                2 │
│ C1          │ P2         │     30 │ 2026-01-10 00:00:00 │                3 │
└─────────────┴────────────┴────────┴─────────────────────┴──────────────────┘



In [7]:
duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")

duckdb.sql("""
CREATE TABLE duolingo_features AS
SELECT t.*, w.difficulty_rank_in_language
FROM duolingo_flagship t
LEFT JOIN read_csv_auto('../../data/word_difficulty.csv') w ON t.lexeme_id = w.lexeme_id
""")

duckdb.sql("SELECT COUNT(*) FROM duolingo_features").show()
duckdb.sql("SELECT COUNT(*) FROM duolingo_features WHERE difficulty_rank_in_language IS NULL").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        16382 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          340 │
└──────────────┘



In [8]:
duckdb.sql("""
SELECT pos, COUNT(*) as missing_difficulty
FROM duolingo_features
WHERE difficulty_rank_in_language IS NULL
GROUP BY pos
ORDER BY missing_difficulty DESC
LIMIT 10
""").show()

┌─────────┬────────────────────┐
│   pos   │ missing_difficulty │
│ varchar │       int64        │
├─────────┼────────────────────┤
│ n       │                136 │
│ vblex   │                128 │
│ adj     │                 33 │
│ adv     │                 14 │
│ pr      │                  6 │
│ vbmod   │                  4 │
│ prn     │                  4 │
│ vbser   │                  4 │
│ num     │                  3 │
│ preadv  │                  2 │
└─────────┴────────────────────┘
  10 rows            2 columns



In [10]:
duckdb.sql("""
SELECT f.pos, 
COUNT(*) as total,
SUM(CASE WHEN f.difficulty_rank_in_language IS NULL THEN 1 ELSE 0 END) as missing,
ROUND(100.0 * SUM(CASE WHEN f.difficulty_rank_in_language IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) as missing_pct
FROM duolingo_features f
GROUP BY f.pos
ORDER BY missing_pct DESC
LIMIT 10
""").show()

┌─────────┬───────┬─────────┬─────────────┐
│   pos   │ total │ missing │ missing_pct │
│ varchar │ int64 │ int128  │   double    │
├─────────┼───────┼─────────┼─────────────┤
│ pprep   │     2 │       1 │        50.0 │
│ vaux    │     7 │       2 │       28.57 │
│ preadv  │    18 │       2 │       11.11 │
│ num     │    39 │       3 │        7.69 │
│ vbmod   │    70 │       4 │        5.71 │
│ vblex   │  2751 │     128 │        4.65 │
│ adj     │   855 │      33 │        3.86 │
│ adv     │   595 │      14 │        2.35 │
│ n       │  7009 │     136 │        1.94 │
│ vbhaver │    63 │       1 │        1.59 │
└─────────┴───────┴─────────┴─────────────┘
  10 rows                       4 columns



In [11]:
duckdb.sql("""
COPY duolingo_features TO '../../data/duolingo_flagship_v5.csv' (HEADER, DELIMITER ',')
""")

duckdb.sql("SELECT COUNT(*) FROM duolingo_features").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        16382 │
└──────────────┘



## Notes

Joined word_difficulty.csv on lexeme_id (LEFT JOIN, preserves all 16,382 rows). 340 rows (~2%) have no difficulty match — checked the missingness by pos, no strong pattern (looks close to MCAR, not tied to a specific word class like grammar_tags was). Left as NULL for now, not imputed — that's a modeling decision for Month 2-3.

Feature set for the flagship going forward: `lag_days`, `history_seen`, `history_correct`, `history_accuracy`, `lag_days_log`, `difficulty_rank_in_language`. Saved as `duolingo_flagship_v5.csv`.